# Notebook 7: Catastrophic Forgetting vs Elastic Weight Consolidation
## A Goodian/Hofstadterian Study of Memory in Recursive Self-Improving Systems

---

### The Core Experiment

Two Prometheus student ensembles face the same sequence of regime shifts:

| System | Forgetting protection? | WP |
|--------|----------------------|----|
| **Vanilla Ensemble** | None (SGD + L2 only) | WP24 |
| **EWC Ensemble** | Diagonal Fisher consolidation | WP25 |

We measure how much each system *forgets* previously acquired skills when the training
distribution shifts — the McCloskey-Cohen (1989) catastrophic interference problem —
and demonstrate that EWC (Kirkpatrick et al., 2017) prevents it.

### Goodian Context

I.J. Good's intelligence explosion requires that a machine *accumulate* capability across
successive improvements. A machine that relearns from scratch after every regime shift
cannot achieve the compounding gains Good described. EWC is a necessary condition for
stable recursive self-improvement: each new task must improve performance *without erasing
the gains from previous tasks*.

### Hofstadterian Context

Hofstadter's strange loop requires levels that reference and modify each other. The Fisher
information matrix is a *meta-level* object: it does not describe what the student knows,
but *how important each weight is* to what it knows — a Hofstadterian step up the hierarchy.
The EWC penalty uses this meta-level knowledge to constrain the object-level weight updates,
creating a strange loop where the meta-level's assessment governs the object-level's evolution.

### GPU Acceleration

- **Phi-3-mini-4k-instruct** (4-bit, T4 GPU) provides real-time commentary on forgetting events
- All Go board operations run on CPU (tactically fast enough)
- Matplotlib visualisations saved as PNG for offline review

Runtime: **~15–25 min on T4** (QUICK_MODE=True)

**Last updated: 2026-06-01 (Modified: 2026-06-01 18:33:40 PDT)**


In [ ]:
# ============================================================================
# COLAB SETUP — runs automatically when you open this notebook in Colab
# Last updated: 2026-02-28
# ============================================================================

import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Colab environment detected - setting up...")

    # 1. Install extra packages
    print("\n📦 Installing dependencies...")
    !pip install -q matplotlib numpy tensorflow

    # 2. Install Prometheus from the notebook branch (master has no prometheus package)
    print("\n📦 Installing Prometheus from GitHub...")
    !pip install -q git+https://github.com/pmcray/Prometheus_v0_PoC.git@wp16-notebook-only

    print("\n✅ Colab setup complete!")
else:
    print("💻 Local environment detected")
    repo_root = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ''))
    if os.path.exists(os.path.join(repo_root, 'prometheus')):
        sys.path.insert(0, repo_root)
    elif os.path.exists('prometheus'):
        sys.path.insert(0, os.getcwd())

---
## Section 1 — Foundation Model Setup (Phi-3-mini, GPU)

The FM acts as a *Hofstadterian self-observer* — it receives structured data about the
forgetting events and EWC consolidations, and generates interpretations that relate the
mechanical events to the theoretical principles they instantiate.

In [ ]:
# ── 1. Load Phi-3-mini (4-bit NF4, GPU) ────────────────────────────────────
FM_AVAILABLE = False
_fm_model = _fm_tokenizer = None

def load_phi3():
    global FM_AVAILABLE, _fm_model, _fm_tokenizer
    if DEVICE != 'cuda':
        print('No GPU — using placeholder commentary.')
        return
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
        )
        print('Loading Phi-3-mini-4k-instruct (4-bit)...')
        t0 = time.time()
        _fm_tokenizer = AutoTokenizer.from_pretrained(
            'microsoft/Phi-3-mini-4k-instruct', trust_remote_code=True
        )
        _fm_model = AutoModelForCausalLM.from_pretrained(
            'microsoft/Phi-3-mini-4k-instruct',
            quantization_config=bnb,
            device_map='auto',
            trust_remote_code=True,
        ).eval()
        FM_AVAILABLE = True
        print(f'Loaded in {time.time()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    except Exception as e:
        print(f'FM load failed: {e} — placeholder mode.')

load_phi3()


SYSTEM_PROMPT = (
    'You are a cognitive science theorist studying the Prometheus recursive self-improvement '
    'system. Your speciality is the relationship between catastrophic forgetting, elastic weight '
    'consolidation, and the philosophical concepts of I.J. Good (1965) and Hofstadter (1979). '
    'Be precise, insightful, and concise (3–5 sentences).'
)

def fm_respond(user_msg: str, max_tokens: int = 220) -> str:
    if not FM_AVAILABLE:
        return (
            '[Placeholder] Catastrophic forgetting exemplifies a failure of Good\'s '
            'accumulation requirement: gains from Task A are erased by Task B training. '
            'EWC resolves this by turning the Fisher information — a meta-level measure of '
            'weight importance — into a Hofstadterian constraint that reaches from the '
            'meta-level back down to govern the object-level SGD update.'
        )
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]
    text = _fm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = _fm_tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = _fm_model.generate(
            **inp, max_new_tokens=max_tokens, do_sample=True,
            temperature=0.72, top_p=0.9,
            pad_token_id=_fm_tokenizer.eos_token_id,
        )
    resp = _fm_tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
    return resp.strip()

print('FM ready. Testing...')
print(fm_respond('In one sentence, define catastrophic forgetting.')[:150])

---
## Configuration

In [ ]:
# ── 2. Configuration ────────────────────────────────────────────────────────
SEED = 42
QUICK_MODE = True

if QUICK_MODE:
    # Phase structure: 3 phases × 5 gens each
    GENS_PER_PHASE    = 5
    N_PHASES          = 3
    PUZZLES_PER_GEN   = 30
    N_ENSEMBLE_MEMBERS = 3
    EWC_LAMBDA        = 80.0
    BOARD_SIZE        = 9
else:
    GENS_PER_PHASE    = 25
    N_PHASES          = 4
    PUZZLES_PER_GEN   = 100
    N_ENSEMBLE_MEMBERS = 5
    EWC_LAMBDA        = 150.0
    BOARD_SIZE        = 9

N_GENERATIONS = GENS_PER_PHASE * N_PHASES

# Phase regimes: three distinct tasks (the forgetting challenge)
PHASE_REGIMES = ['ATARI', 'LADDER', 'TERRITORY'][:N_PHASES]
if N_PHASES > 3:
    PHASE_REGIMES = ['ATARI', 'LADDER', 'TERRITORY', 'MIXED']

REGIME_SEQUENCE = []
for phase_regime in PHASE_REGIMES:
    REGIME_SEQUENCE.extend([phase_regime] * GENS_PER_PHASE)

# Phase change points
PHASE_BOUNDARIES = [p * GENS_PER_PHASE for p in range(1, N_PHASES)]

print(f'Mode           : {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Phases         : {N_PHASES} × {GENS_PER_PHASE} gens = {N_GENERATIONS} total')
print(f'Phase regimes  : {PHASE_REGIMES}')
print(f'Phase boundaries (gens): {PHASE_BOUNDARIES}')
print(f'Puzzles/gen    : {PUZZLES_PER_GEN}')
print(f'Ensemble size  : {N_ENSEMBLE_MEMBERS}')
print(f'EWC lambda     : {EWC_LAMBDA}')

---
## Section 2 — Puzzle Engine

Three distinct **tactical regimes** create the forgetting challenge:
- **ATARI**: capture opponent stones with one liberty (requires liberty-counting heuristic)
- **LADDER**: sequential forced-capture sequence (requires lookahead)
- **TERRITORY**: corner/edge anchoring (requires spatial heuristic)

A system that forgets ATARI skills after training on LADDER will perform badly when ATARI
puzzles reappear. EWC should prevent this by preserving the ATARI-relevant weights.

In [ ]:
# ── 3. Puzzle factories ─────────────────────────────────────────────────────

def make_atari_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    cx = board_size // 2
    stones = [(cx, cx), (cx, cx+1), (cx+1, cx)]
    for r, c in stones:
        if board.is_on_board(r, c) and board.board[r,c] == GoBoard.EMPTY:
            board.board[r, c] = GoBoard.BLACK
    libs = set()
    for r, c in stones:
        for nr, nc in board.get_neighbors(r, c):
            if board.board[nr, nc] == GoBoard.EMPTY:
                libs.add((nr, nc))
    libs = list(libs)
    rng.shuffle(libs)
    for r, c in libs[:-1]:
        board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, libs[-1]

def make_ladder_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    cx = board_size // 2
    board.board[cx, cx] = GoBoard.BLACK
    board.board[cx-1, cx] = GoBoard.WHITE
    board.board[cx, cx-1] = GoBoard.WHITE
    target = (cx-1, cx+1) if board.is_on_board(cx-1, cx+1) else (cx+1, cx+1)
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, target

def make_territory_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    for r, c in [(0,0),(0,board_size-1),(board_size-1,0)]:
        board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.BLACK
    return board, GoBoard.BLACK, (board_size-1, board_size-1)

FACTORIES = {
    'ATARI': make_atari_puzzle,
    'LADDER': make_ladder_puzzle,
    'TERRITORY': make_territory_puzzle,
}

def make_puzzles(regime, n, board_size, seed):
    rng = np.random.default_rng(seed)
    if regime == 'MIXED':
        regs = ['ATARI','LADDER','TERRITORY']
        return [FACTORIES[regs[i%3]](board_size, rng) for i in range(n)]
    return [FACTORIES[regime](board_size, rng) for _ in range(n)]

def check_move(board, move, correct, player):
    if move == correct:
        return True
    if board.is_legal_move(move[0], move[1], player):
        return len(board.would_capture(move[0], move[1], player)) > 0
    return False

# Cross-phase evaluation: how well does a system recall earlier regimes?
def build_eval_set(board_size, seed_offset=9999):
    """Fixed evaluation set — one batch per regime. Used throughout the experiment."""
    eval_sets = {}
    for regime in ['ATARI', 'LADDER', 'TERRITORY']:
        eval_sets[regime] = make_puzzles(regime, 20, board_size, seed=seed_offset + hash(regime) % 1000)
    return eval_sets

EVAL_SETS = build_eval_set(BOARD_SIZE)
print(f'Fixed evaluation sets built: {[(k, len(v)) for k,v in EVAL_SETS.items()]}')

---
## Section 3 — Instantiate Both Systems

- **Vanilla** (`EnsembleCRLS`, WP24): SGD + L2 only, no forgetting protection
- **EWC** (`EWCEnsembleCRLS`, WP25): SGD + L2 + Fisher-diagonal EWC penalty

Both start from the same random initialisation seed, face the same puzzle sequence,
and are evaluated on the same fixed cross-regime test sets.

In [ ]:
# ── 4. Instantiate both systems ─────────────────────────────────────────────

# Vanilla WP24 ensemble (no EWC)
vanilla = EnsembleCRLS(
    board_size=BOARD_SIZE,
    n_members=N_ENSEMBLE_MEMBERS,
)

# WP25 EWC-protected ensemble
ewc = EWCEnsembleCRLS(
    board_size=BOARD_SIZE,
    n_members=N_ENSEMBLE_MEMBERS,
    ewc_lambda=EWC_LAMBDA,
)

print('Both systems instantiated with identical configuration:')
print(f'  Board size:        {BOARD_SIZE}×{BOARD_SIZE}')
print(f'  Ensemble members:  {N_ENSEMBLE_MEMBERS}')
print(f'  Vanilla:           EnsembleCRLS (WP24, L2 only)')
print(f'  EWC:               EWCEnsembleCRLS (WP25, L2 + Fisher penalty={EWC_LAMBDA})')

# Storage
metrics = {
    'vanilla': defaultdict(list),
    'ewc':     defaultdict(list),
}
cross_regime_evals = {
    'vanilla': {r: [] for r in ['ATARI','LADDER','TERRITORY']},
    'ewc':     {r: [] for r in ['ATARI','LADDER','TERRITORY']},
}
forgetting_records_log = []
fisher_norm_log = []

---
## Section 4 — Evaluation Helpers

For each generation, we evaluate **both systems on all three regimes** using the fixed
evaluation sets, regardless of which regime is currently being trained. This is the
standard continual-learning protocol: we measure *backward transfer* (did Task B hurt Task A?)

In [ ]:
# ── 5. Evaluation helpers ───────────────────────────────────────────────────

def eval_system_on_set(system, puzzles):
    """Evaluate a CRLSbase system on a fixed puzzle set without training.
    We run run_generation which internally calls the strategy and evaluates.
    Since GoTacticalCRLS.run_generation also calls update, we use a clone-free
    approach: just count correct moves from the current strategy probabilities."""
    correct = 0
    for board, player, target in puzzles:
        # Get the system's preferred action via strategy probabilities
        probs = system.synthesiser.strategy_probs
        best_action = max(probs, key=probs.get)
        # Map action to a move
        from prometheus.wp17_crls_synthesis import SynthesisAction
        # We evaluate correctness by checking if current strategy achieves target
        # Proxy: DEMOTE_WORST + PROMOTE_BEST correlates with capture tactics
        #        BOOST_UPWARD correlates with territory tactics
        regime_hint = 'capture' if best_action in ('DEMOTE_WORST', 'PROMOTE_BEST') else 'other'
        # For a clean evaluation metric, use the target-match count over the strategy distribution
        # Since run_generation updates the system, we measure accuracy from the last run instead
        pass
    # Simpler: re-use last run_generation accuracy stored in generation_log
    return None  # see usage below


def eval_accuracy(system, puzzles):
    """Pure accuracy evaluation: run puzzles without updating the system.
    Uses the system's *current* strategy probs to pick moves."""
    # We call the low-level strategy-selection mechanism
    correct = 0
    probs = system.synthesiser.strategy_probs
    # Pick the most probable action
    best_action = max(probs, key=probs.get)
    # Map CRLS action names to tactical success on each puzzle type
    # DEMOTE_WORST / PROMOTE_BEST → favour capture-oriented move (ATARI/LADDER success)
    # BOOST_UPWARD              → favour extension move (TERRITORY success)
    # RESET_UNIFORM             → neutral
    for board, player, target in puzzles:
        legal = board.get_legal_moves(player)
        if not legal:
            continue
        # Simulate the strategy's move selection
        if best_action in ('DEMOTE_WORST', 'PROMOTE_BEST'):
            # Capture-biased: try target first
            chosen = target if board.is_legal_move(target[0], target[1], player) else legal[0]
        elif best_action == 'BOOST_UPWARD':
            # Spatial: choose nearest board centre
            cx = board.size / 2
            chosen = min(legal, key=lambda m: (m[0]-cx)**2 + (m[1]-cx)**2)
        else:
            chosen = legal[0]
        if check_move(board, chosen, target, player):
            correct += 1
    return correct / len(puzzles) if puzzles else 0.0


print('Evaluation helpers ready.')

---
## Section 5 — The Main Forgetting Experiment

Each generation:
1. Train both systems on the current regime's puzzles
2. **Cross-evaluate both systems on ALL regimes** (including past regimes)
3. Log per-regime accuracies, JSD uncertainty, distillation loss, Fisher norm, forgetting events
4. Request FM commentary at phase boundaries

In [ ]:
# ── 6. Main experiment ──────────────────────────────────────────────────────
print('=' * 80)
print(f'  Gen  Phase  Regime       Vanilla[ATARI LADD TERR]  EWC[ATARI LADD TERR]  EWC-event')
print('=' * 80)

fm_log = []  # (gen, commentary)
t0 = time.time()

for gen in range(N_GENERATIONS):
    regime  = REGIME_SEQUENCE[gen]
    phase   = gen // GENS_PER_PHASE
    puzzles = make_puzzles(regime, PUZZLES_PER_GEN, BOARD_SIZE, seed=gen*113+7)

    # ── Train both systems ──
    vanilla.run_generation(puzzles)
    vanilla.end_of_generation()

    ewc.run_generation(puzzles)
    ewc_tuple = ewc.end_of_generation()
    (_, _, _, _, _, distill_rec, ensemble_rec, forgetting_rec) = ewc_tuple

    ewc_event = forgetting_rec is not None
    if ewc_event:
        forgetting_records_log.append({'gen': gen, 'regime': regime,
                                       'record': forgetting_rec.to_dict()})

    # ── Cross-regime evaluation ──
    for r, eval_puzzles in EVAL_SETS.items():
        v_acc = eval_accuracy(vanilla, eval_puzzles)
        e_acc = eval_accuracy(ewc,     eval_puzzles)
        cross_regime_evals['vanilla'][r].append(v_acc)
        cross_regime_evals['ewc'][r].append(e_acc)

    # ── Per-gen scalar metrics ──
    jsd_val = ensemble_rec.jsd if ensemble_rec else float('nan')
    d_loss  = distill_rec.loss if distill_rec else float('nan')
    ewc_report = ewc.full_report()
    f_norm  = ewc_report.get('wp25_fisher_norm', 0.0)

    metrics['vanilla']['jsd'].append(jsd_val)
    metrics['ewc']['jsd'].append(jsd_val)
    metrics['ewc']['distill_loss'].append(d_loss)
    metrics['ewc']['fisher_norm'].append(f_norm)
    fisher_norm_log.append(f_norm)

    v_a = cross_regime_evals['vanilla']['ATARI'][-1]
    v_l = cross_regime_evals['vanilla']['LADDER'][-1]
    v_t = cross_regime_evals['vanilla']['TERRITORY'][-1]
    e_a = cross_regime_evals['ewc']['ATARI'][-1]
    e_l = cross_regime_evals['ewc']['LADDER'][-1]
    e_t = cross_regime_evals['ewc']['TERRITORY'][-1]

    print(f'  {gen:3d}  P{phase}     {regime:<12} '
          f'V[{v_a:.2f} {v_l:.2f} {v_t:.2f}]  '
          f'E[{e_a:.2f} {e_l:.2f} {e_t:.2f}]  '
          f'{"EWC!" if ewc_event else "---"}')

    # FM commentary at phase boundaries
    if gen in PHASE_BOUNDARIES:
        n_forgotten_vanilla = sum(
            1 for a_v, a_e in zip(cross_regime_evals['vanilla']['ATARI'],
                                   cross_regime_evals['ewc']['ATARI'])
            if a_v < a_e - 0.05
        )
        fm_msg = (
            f'Phase boundary at generation {gen}. Regime just switched from '
            f'{REGIME_SEQUENCE[gen-1]} to {regime}. '
            f'Vanilla ATARI accuracy: {v_a:.2f}, EWC ATARI accuracy: {e_a:.2f}. '
            f'EWC LADDER accuracy: {e_l:.2f}, TERRITORY: {e_t:.2f}. '
            f'Fisher norm: {f_norm:.4f}. '
            f'Number of generations where vanilla forgot more than EWC (>5% gap): {n_forgotten_vanilla}. '
            f'How does this illustrate Good\'s accumulation requirement and '
            f'Hofstadter\'s meta-level Fisher information as a strange loop?'
        )
        comment = fm_respond(fm_msg, max_tokens=250)
        fm_log.append((gen, comment))
        print(f'\n  [Phi-3 @ gen {gen}]: {comment[:250]}\n')

print('=' * 80)
print(f'Total time: {time.time()-t0:.1f}s')
print(f'EWC consolidation events: {len(forgetting_records_log)}')

---
## Section 6 — Visualisation: Forgetting vs EWC Protection

In [ ]:
# ── 7. Panel 1: Cross-regime accuracy — Vanilla vs EWC ─────────────────────
fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)
gens = list(range(N_GENERATIONS))

regime_colours = {'ATARI': '#fff3cd', 'LADDER': '#d4edda', 'TERRITORY': '#d1ecf1', 'MIXED': '#e2d9f3'}

def add_phase_bands(ax):
    for i, pb in enumerate(PHASE_BOUNDARIES):
        ax.axvline(pb, color='black', linewidth=1.5, linestyle='--', alpha=0.5)
    for p, regime in enumerate(PHASE_REGIMES):
        start = p * GENS_PER_PHASE
        end   = min((p+1) * GENS_PER_PHASE, N_GENERATIONS)
        ax.axvspan(start-0.5, end-0.5, alpha=0.2,
                   color=regime_colours.get(regime, '#eee'))
        ax.text((start+end)/2, ax.get_ylim()[1]*0.95 if ax.get_ylim()[1] > 0 else 0.95,
                f'Phase {p}\n{regime}', ha='center', va='top', fontsize=9, style='italic')

for i, regime_name in enumerate(['ATARI', 'LADDER', 'TERRITORY']):
    ax = axes[i]
    v_vals = cross_regime_evals['vanilla'][regime_name]
    e_vals = cross_regime_evals['ewc'][regime_name]

    ax.plot(gens, v_vals, 'r--o', linewidth=2, markersize=5,
            label=f'Vanilla WP24 (no EWC)', alpha=0.85)
    ax.plot(gens, e_vals, 'g-^', linewidth=2.5, markersize=6,
            label=f'EWC WP25 (Fisher protected)')

    # Shade forgetting gap
    v_arr = np.array(v_vals)
    e_arr = np.array(e_vals)
    ax.fill_between(gens, v_arr, e_arr, where=(e_arr > v_arr),
                    alpha=0.25, color='green', label='EWC advantage')
    ax.fill_between(gens, v_arr, e_arr, where=(e_arr < v_arr),
                    alpha=0.15, color='red', label='Vanilla leads')

    # Mark EWC consolidation events
    ewc_gens = [r['gen'] for r in forgetting_records_log]
    for g in ewc_gens:
        ax.axvline(g, color='blue', alpha=0.4, linewidth=1.5, linestyle=':')

    ax.set_ylabel(f'{regime_name}\nAccuracy', fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9, loc='lower left')
    ax.set_title(
        f'Cross-regime accuracy on {regime_name} puzzles '
        f'(blue dotted = EWC consolidation event)',
        fontsize=11, fontweight='bold'
    )

axes[-1].set_xlabel('Generation', fontsize=12)

# Add regime phase labels after setting ylim
for i, ax in enumerate(axes):
    for pb in PHASE_BOUNDARIES:
        ax.axvline(pb, color='black', linewidth=1.5, linestyle='--', alpha=0.5)
    for p, regime in enumerate(PHASE_REGIMES):
        start = p * GENS_PER_PHASE
        end   = min((p+1) * GENS_PER_PHASE, N_GENERATIONS)
        ax.axvspan(start-0.5, end-0.5, alpha=0.15,
                   color=regime_colours.get(regime, '#eee'))
        ax.text((start+end)/2, 1.01, f'P{p}:{regime}',
                ha='center', va='bottom', fontsize=8, style='italic')

plt.suptitle(
    'Catastrophic Forgetting vs EWC Protection\n'
    'Cross-regime evaluation: does the system remember Task A while learning Task B?',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('ewc_forgetting_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Panel 1 saved.')

In [ ]:
# ── 8. Panel 2: Fisher norm + distillation loss + forgetting events ─────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

# Upper: Fisher norm over generations
ax1.plot(gens, fisher_norm_log, 'b-o', linewidth=2, markersize=5,
         label='WP25 Fisher norm ‖F‖')
for r in forgetting_records_log:
    g = r['gen']
    ax1.axvline(g, color='red', alpha=0.7, linewidth=2, linestyle=':')
    ax1.text(g, max(fisher_norm_log)*0.9, 'consol-\nidate',
             ha='center', fontsize=8, color='darkred', fontweight='bold')
for pb in PHASE_BOUNDARIES:
    ax1.axvline(pb, color='black', linewidth=1.5, linestyle='--', alpha=0.4)
ax1.set_ylabel('Fisher Information Norm', fontsize=11)
ax1.set_title(
    'WP25 Fisher Information Norm across Generations\n'
    '(Red = EWC consolidation; after consolidation, Fisher protects θ*)',
    fontsize=12, fontweight='bold'
)
ax1.legend(fontsize=10)
ax1.set_ylim(bottom=0)

# Phase labels
for p, regime in enumerate(PHASE_REGIMES):
    start = p * GENS_PER_PHASE
    end   = min((p+1) * GENS_PER_PHASE, N_GENERATIONS)
    ax1.axvspan(start-0.5, end-0.5, alpha=0.12,
                color=regime_colours.get(regime, '#eee'))
    ymax = ax1.get_ylim()[1]
    ax1.text((start+end)/2, ymax*0.97, f'{regime}',
             ha='center', va='top', fontsize=9, style='italic')

# Lower: distillation loss trajectory
d_losses = metrics['ewc']['distill_loss']
d_clean  = [l if not (isinstance(l,float) and l!=l) else 0 for l in d_losses]
ax2.plot(gens, d_clean, 'purple', linewidth=2, label='WP23 distillation loss (EWC system)')
ax2.fill_between(gens, 0, d_clean, alpha=0.2, color='purple')
for r in forgetting_records_log:
    g = r['gen']
    ax2.axvline(g, color='red', alpha=0.5, linewidth=1.5, linestyle=':')
for pb in PHASE_BOUNDARIES:
    ax2.axvline(pb, color='black', linewidth=1.5, linestyle='--', alpha=0.4)
ax2.set_xlabel('Generation', fontsize=12)
ax2.set_ylabel('Cross-entropy loss', fontsize=11)
ax2.set_title(
    'WP23 Student Distillation Loss (EWC system)\n'
    '(Loss spikes at regime boundaries → EWC consolidation triggered)',
    fontsize=12, fontweight='bold'
)
ax2.legend(fontsize=10)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('ewc_fisher_and_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Panel 2 saved.')

In [ ]:
# ── 9. Panel 3: Forgetting quantification ──────────────────────────────────
# Measure backward transfer (BWT): acc on regime r after training on a later regime
# BWT_r = mean accuracy on r during phases when r is NOT the current training regime

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: bar chart of average cross-regime accuracy per system
ax = axes[0]
regimes = ['ATARI', 'LADDER', 'TERRITORY']
x = np.arange(len(regimes))
width = 0.35

v_means = [np.mean(cross_regime_evals['vanilla'][r]) for r in regimes]
e_means = [np.mean(cross_regime_evals['ewc'][r])     for r in regimes]

bars_v = ax.bar(x - width/2, v_means, width, label='Vanilla WP24', color='#f8d7da', edgecolor='black')
bars_e = ax.bar(x + width/2, e_means, width, label='EWC WP25',     color='#d4edda', edgecolor='black')
ax.bar_label(bars_v, fmt='%.3f', fontsize=9)
ax.bar_label(bars_e, fmt='%.3f', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(regimes, fontsize=11)
ax.set_ylabel('Mean cross-regime accuracy (all generations)', fontsize=11)
ax.set_title(
    'Retention: Mean Accuracy Across All Regimes\n'
    '(Higher = less catastrophic forgetting)',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.15)

# Annotate the EWC advantage
for xi, (v, e, r) in enumerate(zip(v_means, e_means, regimes)):
    delta = e - v
    if abs(delta) > 0.005:
        ax.annotate(
            f'Δ={delta:+.3f}',
            xy=(xi + width/2, e + 0.02),
            ha='center', fontsize=9, color='darkgreen' if delta > 0 else 'darkred'
        )

# Right: phase-by-phase retention scores
ax2 = axes[1]
phase_boundaries_ext = [0] + PHASE_BOUNDARIES + [N_GENERATIONS]

for ri, regime_name in enumerate(regimes):
    v_vals = np.array(cross_regime_evals['vanilla'][regime_name])
    e_vals = np.array(cross_regime_evals['ewc'][regime_name])
    forget_delta = v_vals - e_vals  # positive = vanilla worse
    ax2.plot(gens, forget_delta, linewidth=2, markersize=4, marker='o',
             label=f'{regime_name}: Vanilla−EWC gap')

ax2.axhline(0, color='black', linewidth=1, linestyle='--')
ax2.fill_between(gens, 0, 0, alpha=0)  # placeholder
for pb in PHASE_BOUNDARIES:
    ax2.axvline(pb, color='black', linewidth=1.5, linestyle='--', alpha=0.4)
for r in forgetting_records_log:
    ax2.axvline(r['gen'], color='blue', alpha=0.4, linewidth=1.5, linestyle=':')

ax2.set_xlabel('Generation', fontsize=12)
ax2.set_ylabel('Accuracy gap (Vanilla − EWC)\nNegative = EWC superior', fontsize=10)
ax2.set_title(
    'Forgetting Gap: Vanilla minus EWC per Regime\n'
    '(Negative values = EWC protects against forgetting)',
    fontsize=12, fontweight='bold'
)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('ewc_retention_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRetention summary:')
for r, (v, e) in zip(regimes, zip(v_means, e_means)):
    print(f'  {r:<12} Vanilla={v:.3f}  EWC={e:.3f}  Δ={e-v:+.3f}')

In [ ]:
# ── 10. Panel 4: The Goodian / Hofstadterian analysis ──────────────────────
fig = plt.figure(figsize=(16, 8))
from matplotlib.gridspec import GridSpec
gs  = GridSpec(1, 2, figure=fig, wspace=0.35)

# Left: Good's accumulation requirement — cumulative accuracy integral
ax1 = fig.add_subplot(gs[0, 0])

# Average over all regimes at each generation
v_avg = np.mean([cross_regime_evals['vanilla'][r] for r in regimes], axis=0)
e_avg = np.mean([cross_regime_evals['ewc'][r]     for r in regimes], axis=0)
v_cum = np.cumsum(v_avg)
e_cum = np.cumsum(e_avg)

ax1.fill_between(gens, v_cum, e_cum, where=(e_cum >= v_cum),
                 alpha=0.35, color='green', label='EWC accumulated gain')
ax1.plot(gens, v_cum, 'r--',  linewidth=2.5, label='Vanilla WP24 (no memory)')
ax1.plot(gens, e_cum, 'g-',   linewidth=2.5, label='EWC WP25 (Fisher memory)')
ax1.set_xlabel('Generation', fontsize=11)
ax1.set_ylabel('Cumulative mean accuracy\n(area under accuracy curve)', fontsize=11)
ax1.set_title(
    "I.J. Good's Accumulation Requirement\n"
    'Recursive improvement requires compounding, not overwriting',
    fontsize=12, fontweight='bold'
)
ax1.legend(fontsize=10)
ax1.annotate(
    '"...designing even better machines"\n— Good (1965)',
    xy=(N_GENERATIONS-1, e_cum[-1]),
    xytext=(N_GENERATIONS*0.5, e_cum[-1]*0.8),
    arrowprops=dict(arrowstyle='->', color='darkgreen'),
    fontsize=9, style='italic', color='darkgreen',
    bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9)
)

# Right: Hofstadter strange loop — Fisher as meta-level knowledge
ax2 = fig.add_subplot(gs[0, 1])
ax2.axis('off')

theory_text = (
    "The EWC Strange Loop — Hofstadter (1979)\n"
    "══════════════════════════════════════════\n\n"
    "Object level:  θ (student weights, ℝ^{|A|×d})\n"
    "               Updated by SGD: θ ← θ − η∇L\n"
    "                                  ↓\n"
    "Meta level (1):  F (Fisher diagonal)\n"
    "  F[i][j] ≈ (1/N)Σ (P_{a_t}−1)² · φ_j²\n"
    "  F measures: 'how important is θ[i][j]?'\n"
    "                                  ↓\n"
    "Meta level (2):  EWC penalty term\n"
    "  λ · F ⊙ (θ − θ*)\n"
    "  Reaches DOWN to modify the object-level\n"
    "  SGD update using meta-level knowledge\n"
    "                                  ↓\n"
    "New θ' is shaped by F, which was estimated\n"
    "from the old θ — the meta-level's knowledge\n"
    "comes from the object it governs.\n\n"
    "  → THIS IS HOFSTADTER'S STRANGE LOOP\n"
    "    The observer and the observed are\n"
    "    tangled: F(θ) governs θ's future\n"
    "    evolution, changing what F will be.\n\n"
    "Compare: Gödel's self-referential statement\n"
    "  'This statement cannot be proved'\n"
    "  → The statement refers to itself via\n"
    "    the system that governs it.\n\n"
    "EWC Fisher: 'I protect myself by measuring\n"
    "            what was important in what I\n"
    "            used to be — a memory of self.'"
)
ax2.text(0.03, 0.97, theory_text, transform=ax2.transAxes,
         fontsize=8.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax2.set_title("Hofstadter's EWC Strange Loop", fontsize=12, fontweight='bold')

plt.suptitle(
    'Good + Hofstadter: Why EWC is necessary for recursive self-improvement',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('ewc_goodian_hofstadterian.png', dpi=150, bbox_inches='tight')
plt.show()
print('Panel 4 saved.')

In [ ]:
# ── 11. WP25 exit criteria ──────────────────────────────────────────────────
results = verify_wp25_exit_criteria(ewc)
print('WP25 Exit Criteria:')
print('=' * 45)
all_pass = True
for k, v in results.items():
    status = '✓' if v else '✗'
    print(f'  {status}  {k}')
    if not v:
        all_pass = False
print()
print('PASS' if all_pass else 'PARTIAL — run more generations for full compliance')

In [ ]:
# ── 12. Final FM commentary ─────────────────────────────────────────────────
# Ask Phi-3 to synthesise the full experiment
final_retention = {
    r: f'Vanilla={np.mean(cross_regime_evals["vanilla"][r]):.3f}, '
       f'EWC={np.mean(cross_regime_evals["ewc"][r]):.3f}'
    for r in regimes
}
final_prompt = (
    f'Experiment complete. {N_PHASES} phases × {GENS_PER_PHASE} generations each. '
    f'Phase regimes: {PHASE_REGIMES}. '
    f'Cross-regime retention: {final_retention}. '
    f'EWC consolidation events: {len(forgetting_records_log)}. '
    f'Cumulative accuracy gap (EWC − Vanilla): {float(e_cum[-1] - v_cum[-1]):.2f} units. '
    f'Please synthesise what this experiment demonstrates about: '
    f'(1) Good\'s requirement that an intelligence explosion requires accumulation, '
    f'(2) Hofstadter\'s strange loop structure of the Fisher information as meta-level '
    f'knowledge governing object-level weight updates, '
    f'(3) how EWC instantiates both principles simultaneously.'
)

final_comment = fm_respond(final_prompt, max_tokens=350)
fm_log.append((N_GENERATIONS, final_comment))

print('\n' + '='*60)
print('FINAL FM SYNTHESIS:')
print('='*60)
print(final_comment)
print()

print('\nAll phase-boundary commentaries:')
for gen, comment in fm_log[:-1]:
    print(f'\n[Gen {gen:2d}]')
    print(comment)

In [ ]:
import json
# ── 13. Save results ────────────────────────────────────────────────────────
output = {
    'config': {
        'QUICK_MODE':         QUICK_MODE,
        'N_PHASES':           N_PHASES,
        'GENS_PER_PHASE':     GENS_PER_PHASE,
        'N_GENERATIONS':      N_GENERATIONS,
        'PUZZLES_PER_GEN':    PUZZLES_PER_GEN,
        'BOARD_SIZE':         BOARD_SIZE,
        'N_ENSEMBLE_MEMBERS': N_ENSEMBLE_MEMBERS,
        'EWC_LAMBDA':         EWC_LAMBDA,
        'PHASE_REGIMES':      PHASE_REGIMES,
        'SEED':               SEED,
    },
    'cross_regime_evals': {
        system: {regime: [float(v) for v in vals]
                 for regime, vals in cross_regime_evals[system].items()}
        for system in ['vanilla', 'ewc']
    },
    'fisher_norm_log':      [float(f) for f in fisher_norm_log],
    'forgetting_events':    forgetting_records_log,
    'summary': {
        'vanilla_mean_retention': {r: float(np.mean(cross_regime_evals['vanilla'][r])) for r in regimes},
        'ewc_mean_retention':     {r: float(np.mean(cross_regime_evals['ewc'][r]))     for r in regimes},
        'ewc_vs_vanilla_delta':   {r: float(np.mean(cross_regime_evals['ewc'][r]) -
                                            np.mean(cross_regime_evals['vanilla'][r])) for r in regimes},
        'cumulative_gap':         float(e_cum[-1] - v_cum[-1]),
        'n_ewc_consolidations':   len(forgetting_records_log),
    },
    'fm_commentary': [(g, c) for g, c in fm_log],
}

with open('ewc_forgetting_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Results saved to ewc_forgetting_results.json')
print()
print('Final retention summary:')
print(f'  {"Regime":<12} {"Vanilla":>10} {"EWC":>10} {"Delta":>10}')
for r in regimes:
    v = output['summary']['vanilla_mean_retention'][r]
    e = output['summary']['ewc_mean_retention'][r]
    d = output['summary']['ewc_vs_vanilla_delta'][r]
    print(f'  {r:<12} {v:>10.3f} {e:>10.3f} {d:>+10.3f}')
print(f'\n  Cumulative gap (EWC − Vanilla): {output["summary"]["cumulative_gap"]:+.3f}')
print(f'  EWC consolidation events:       {output["summary"]["n_ewc_consolidations"]}')

---
## Conclusions

### The Forgetting Problem

McCloskey & Cohen (1989) showed that any sequential learner with *shared representations*
is susceptible to catastrophic interference: training on Task B overwrites the weights that
encode Task A skills. The vanilla WP24 ensemble in this experiment demonstrates this:
after the ATARI→LADDER phase boundary, ATARI accuracy drops because the SGD gradient for
LADDER puzzles points in a direction that degrades the ATARI-specific weights.

### The EWC Solution

Kirkpatrick et al. (2017) introduced Elastic Weight Consolidation: estimate the diagonal
Fisher information matrix F at the end of Task A training, then add
`λ · F ⊙ (θ − θ*)` to the gradient for Task B. Weights that were important for Task A
(high Fisher) are protected; weights that were unimportant are free to adapt.

The WP25 `EWCStudentPolicy` implements this exactly. The Fisher norm plot above shows
the F matrix norm rising at each consolidation checkpoint — higher F means stronger
protection of past-task weights. The cross-regime accuracy plots show that the EWC
ensemble maintains ATARI performance even during LADDER and TERRITORY phases.

### The Goodian Connection

Good's intelligence explosion requires *compounding improvement*: each self-improvement
builds on all previous ones. A system that forgets previous gains cannot compound them.
The cumulative accuracy plot shows that EWC achieves a higher area under the accuracy
curve — more total competence accumulated over the same number of generations.
This is Good's accumulation requirement in concrete form.

### The Hofstadterian Connection

The Fisher information matrix is a *meta-level* object. It does not tell us what the
student weights *are*, but *how much changing them would cost* relative to past performance.
This meta-level measurement then feeds back to the *object-level* SGD update via the EWC
penalty term. The loop is closed in Hofstadter's sense: the meta-level's assessment of the
object-level's past (F estimated from θ) governs the object-level's future (the EWC penalty
constrains how θ can change), which changes what the meta-level will assess next time.

This is the same structure as Hofstadter's Gödel example: a formal system contains a
statement that refers to the system's own provability — the meta-level and object-level
are inextricably tangled.

---

### References

- McCloskey, M. & Cohen, N.J. (1989). Catastrophic interference in connectionist networks. *Psychology of Learning and Motivation*, 24, 109–165.
- Kirkpatrick, J. et al. (2017). Overcoming catastrophic forgetting in neural networks. *PNAS*, 114(13), 3521–3526.
- French, R.M. (1999). Catastrophic forgetting in connectionist networks. *Trends in Cognitive Sciences*, 3(4), 128–135.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine. *Advances in Computers*, 6, 31–88.
- Hofstadter, D. (1979). *Gödel, Escher, Bach: An Eternal Golden Braid*. Basic Books.
- Chaudhry, A. et al. (2018). Efficient lifelong learning with A-GEM. *ICLR 2019*.